# Phase 2 — Data Cleaning & Integration
**Project:** Global Food Security Risk Analysis  
**Notebook:** 02_cleaning.ipynb  

## Objectives
1. Load and clean all four raw datasets
2. Standardize country codes (M49 → ISO3)
3. Merge all datasets into a single analysis-ready file
4. Save cleaned files to `data/processed/`

## Datasets
| File | Source | Contents |
|---|---|---|
| `Suite_of_Food_Security_Indicators.csv` | FAOSTAT | Undernourishment rate & 4 other indicators |
| `FoodBalances.csv` | FAOSTAT | Protein, fat, kcal supply |
| `ProductionIndices.csv` | FAOSTAT | Agriculture & food production index (per capita) |
| `WorldDevelopmentIndicators.csv` | World Bank | GDP, population, precipitation, poverty |

---
## Cell 1 — Import Libraries & Load Raw Data

In [1]:
import pandas as pd
import numpy as np
import re
import os
import pycountry

# Load all four raw datasets
# encoding='utf-8-sig' strips the BOM character (ï»¿) found in FAOSTAT exports
fsi = pd.read_csv("../data/raw/Suite_of_Food_Security_Indicators.csv", encoding="utf-8-sig")
fb  = pd.read_csv("../data/raw/FoodBalances.csv",                       encoding="utf-8-sig")
pi  = pd.read_csv("../data/raw/ProductionIndices.csv",                  encoding="utf-8-sig")
wdi = pd.read_csv("../data/raw/WorldDevelopmentIndicators.csv",         encoding="utf-8-sig")

print("All files loaded successfully")
for name, df in [("FSI", fsi), ("FB", fb), ("PI", pi), ("WDI", wdi)]:
    print(f"  {name}: {df.shape}")

All files loaded successfully
  FSI: (20471, 15)
  FB: (7275, 15)
  PI: (4506, 14)
  WDI: (1601, 17)


---
## Cell 2 — Helper Function: Year Parser

**Problem discovered during exploration:**  
Some FAOSTAT indicators (e.g. undernourishment rate) use **3-year rolling averages**,  
so their `Year` column contains range strings like `'2009-2011'` instead of `'2010'`.  
Standard `pd.to_numeric()` converts these to `NaN`, causing all rows to be dropped.

**Solution:** Extract the **end year** from range strings.
- `'2009-2011'` → `2011`
- `'2010'` → `2010`

In [2]:
def parse_year(y):
    """
    Parse FAOSTAT year strings into integers.
    Handles both single years ('2010') and 3-year average ranges ('2009-2011').
    For range strings, the end year is used (e.g. '2009-2011' -> 2011).
    """
    y = str(y).strip()
    if "-" in y:
        return int(y.split("-")[-1])  # '2009-2011' -> 2011
    return int(y)

print("parse_year() defined")
for test in ["2010", "2009-2011", "2020-2022"]:
    print(f"  parse_year('{test}') -> {parse_year(test)}")

parse_year() defined
  parse_year('2010') -> 2010
  parse_year('2009-2011') -> 2011
  parse_year('2020-2022') -> 2022


---
## Cell 3 — Clean FSI (Food Security Indicators)

**Structure:** Long format — one row per (country, year, indicator)  
**Country code:** M49  

**Key issues:**
- `Year` column contains range strings (e.g. `'2009-2011'`) for 3-year average indicators
- `Value` column contains `'<2.5'` strings → replaced with `2.5` (conservative upper bound)

**Indicators selected:**
| Column Name | Original Indicator |
|---|---|
| `undernourishment_pct` | Prevalence of undernourishment (%) — **TARGET VARIABLE** |
| `dietary_energy_supply_kcal` | Dietary energy supply (kcal/cap/day) |
| `food_supply_variability_kcal` | Per capita food supply variability (kcal/cap/day) |
| `cereal_import_dependency_pct` | Cereal import dependency ratio (%) |
| `political_stability_index` | Political stability and absence of violence (index) |

**Excluded:**
- GDP per capita (duplicated in WDI)
- Moderate/severe food insecurity indicators (too correlated with target — leakage risk)

In [3]:
FSI_ITEMS = [
    "Prevalence of undernourishment (percent) (3-year average)",
    "Dietary energy supply used in the estimation of the prevalence of undernourishment (kcal/cap/day)",
    "Per capita food supply variability (kcal/cap/day)",
    "Cereal import dependency ratio (percent) (3-year average)",
    "Political stability and absence of violence/terrorism (index)",
]

fsi_work = fsi[fsi["Item"].isin(FSI_ITEMS)].copy()

# Parse year column (handles both '2010' and '2009-2011' formats)
fsi_work["Year"] = fsi_work["Year"].apply(parse_year)

# Fix Value column:
# '<2.5' appears where the true value is below the detection limit.
# Replaced with 2.5 as a conservative upper bound.
fsi_work["Value"] = fsi_work["Value"].astype(str).str.strip().replace("<2.5", "2.5")
fsi_work["Value"] = pd.to_numeric(fsi_work["Value"], errors="coerce")

# Filter to 2010-2022
fsi_work = fsi_work[fsi_work["Year"].between(2010, 2022)]

# Keep only necessary columns
fsi_work = fsi_work[["Area Code (M49)", "Area", "Year", "Item", "Value"]]

# Aggregate duplicates by mean (some indicators have overlapping 3-year windows)
fsi_work = fsi_work.groupby(
    ["Area Code (M49)", "Area", "Year", "Item"], as_index=False
)["Value"].mean()

# Pivot: long -> wide (Item becomes columns)
fsi_clean = fsi_work.pivot_table(
    index=["Area Code (M49)", "Area", "Year"],
    columns="Item",
    values="Value"
).reset_index()
fsi_clean.columns.name = None

# Rename columns to short, descriptive names
fsi_clean = fsi_clean.rename(columns={
    "Prevalence of undernourishment (percent) (3-year average)":
        "undernourishment_pct",
    "Dietary energy supply used in the estimation of the prevalence of undernourishment (kcal/cap/day)":
        "dietary_energy_supply_kcal",
    "Per capita food supply variability (kcal/cap/day)":
        "food_supply_variability_kcal",
    "Cereal import dependency ratio (percent) (3-year average)":
        "cereal_import_dependency_pct",
    "Political stability and absence of violence/terrorism (index)":
        "political_stability_index",
})

print(f"FSI cleaned: {fsi_clean.shape}")
print(f"  Countries : {fsi_clean['Area'].nunique()}")
print(f"  Years     : {sorted(fsi_clean['Year'].unique())}")
print(f"  Columns   : {fsi_clean.columns.tolist()}")
print()
print("Note: NaN in year 2010 for 3-year average indicators is expected.")
print("      '2009-2011' maps to 2011, so no entry exists for 2010.")
print(fsi_clean.head(3).to_string())

FSI cleaned: (2612, 8)
  Countries : 201
  Years     : [2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022]
  Columns   : ['Area Code (M49)', 'Area', 'Year', 'cereal_import_dependency_pct', 'dietary_energy_supply_kcal', 'food_supply_variability_kcal', 'political_stability_index', 'undernourishment_pct']

Note: NaN in year 2010 for 3-year average indicators is expected.
      '2009-2011' maps to 2011, so no entry exists for 2010.
   Area Code (M49)         Area  Year  cereal_import_dependency_pct  dietary_energy_supply_kcal  food_supply_variability_kcal  political_stability_index  undernourishment_pct
0                4  Afghanistan  2010                           NaN                      2200.0                          28.0                      -2.58                   NaN
1                4  Afghanistan  2011                          29.6                      2172.0                          28.0                      -2.50                  17.7
2               

---
## Cell 4 — Clean FB (Food Balances)

**Structure:** Long format  
**Key difference from FSI:** Indicators are stored in the `Element` column (not `Item`).  
`Item` only contains `'Grand Total'` (national aggregate).  

**Indicators used (all 3):**
| Column Name | Original Element |
|---|---|
| `food_supply_kcal` | Food supply (kcal/capita/day) |
| `protein_supply_g` | Protein supply quantity (g/capita/day) |
| `fat_supply_g` | Fat supply quantity (g/capita/day) |

In [4]:
fb_work = fb.copy()

# Parse year (FB uses single-year format, but use same function for consistency)
fb_work["Year"] = fb_work["Year"].apply(parse_year)

# Coerce Value to numeric
fb_work["Value"] = pd.to_numeric(fb_work["Value"], errors="coerce")

# Filter to 2010-2022
fb_work = fb_work[fb_work["Year"].between(2010, 2022)]

# Keep only necessary columns (pivot on Element, not Item)
fb_work = fb_work[["Area Code (M49)", "Area", "Year", "Element", "Value"]]

# Aggregate duplicates by mean
fb_work = fb_work.groupby(
    ["Area Code (M49)", "Area", "Year", "Element"], as_index=False
)["Value"].mean()

# Pivot: long -> wide (Element becomes columns)
fb_clean = fb_work.pivot_table(
    index=["Area Code (M49)", "Area", "Year"],
    columns="Element",
    values="Value"
).reset_index()
fb_clean.columns.name = None

# Rename columns
fb_clean = fb_clean.rename(columns={
    "Food supply (kcal/capita/day)":           "food_supply_kcal",
    "Protein supply quantity (g/capita/day)":  "protein_supply_g",
    "Fat supply quantity (g/capita/day)":      "fat_supply_g",
})

print(f"FB cleaned: {fb_clean.shape}")
print(f"  Countries : {fb_clean['Area'].nunique()}")
print(f"  Years     : {sorted(fb_clean['Year'].unique())}")
print(f"  Columns   : {fb_clean.columns.tolist()}")
print(fb_clean.head(3).to_string())

FB cleaned: (2248, 6)
  Countries : 179
  Years     : [2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022]
  Columns   : ['Area Code (M49)', 'Area', 'Year', 'fat_supply_g', 'food_supply_kcal', 'protein_supply_g']
   Area Code (M49)         Area  Year  fat_supply_g  food_supply_kcal  protein_supply_g
0                4  Afghanistan  2010         37.93           2200.21             65.54
1                4  Afghanistan  2011         35.86           2171.86             63.74
2                4  Afghanistan  2012         37.34           2165.88             62.67


---
## Cell 5 — Clean PI (Production Indices)

**Structure:** Long format  
**Key notes:**
- `Item` has 2 values: `'Agriculture'` and `'Food'` → both used as separate columns
- `Element` has only 1 value (gross per capita production index)
- Years **2015 and 2018 are missing from the source data** (confirmed, not a processing error)
- Year 2023 present in source → excluded (project scope: 2010–2022)

**Indicators used:**
| Column Name | Original Item |
|---|---|
| `agri_production_index` | Agriculture |
| `food_production_index` | Food |

In [5]:
pi_work = pi.copy()

# Parse year
pi_work["Year"] = pi_work["Year"].apply(parse_year)

# Coerce Value to numeric
pi_work["Value"] = pd.to_numeric(pi_work["Value"], errors="coerce")

# Filter to 2010-2022 (excludes 2023 which exists in source)
pi_work = pi_work[pi_work["Year"].between(2010, 2022)]

# Keep only necessary columns (pivot on Item)
pi_work = pi_work[["Area Code (M49)", "Area", "Year", "Item", "Value"]]

# Aggregate duplicates by mean
pi_work = pi_work.groupby(
    ["Area Code (M49)", "Area", "Year", "Item"], as_index=False
)["Value"].mean()

# Pivot: long -> wide (Item becomes columns)
pi_clean = pi_work.pivot_table(
    index=["Area Code (M49)", "Area", "Year"],
    columns="Item",
    values="Value"
).reset_index()
pi_clean.columns.name = None

# Rename columns
pi_clean = pi_clean.rename(columns={
    "Agriculture": "agri_production_index",
    "Food":        "food_production_index",
})

print(f"PI cleaned: {pi_clean.shape}")
print(f"  Countries : {pi_clean['Area'].nunique()}")
print(f"  Years     : {sorted(pi_clean['Year'].unique())}")
print(f"  Columns   : {pi_clean.columns.tolist()}")
print()
print("Note: Years 2015 and 2018 are absent from the original source data.")
print(pi_clean.head(3).to_string())

PI cleaned: (2082, 5)
  Countries : 200
  Years     : [2010, 2011, 2012, 2013, 2014, 2016, 2017, 2019, 2020, 2021, 2022]
  Columns   : ['Area Code (M49)', 'Area', 'Year', 'agri_production_index', 'food_production_index']

Note: Years 2015 and 2018 are absent from the original source data.
   Area Code (M49)         Area  Year  agri_production_index  food_production_index
0                4  Afghanistan  2010                 109.33                 109.42
1                4  Afghanistan  2011                 100.63                 100.53
2                4  Afghanistan  2012                 107.26                 107.34


---
## Cell 6 — Clean WDI (World Bank Development Indicators)

**Structure:** Wide format — each year is a separate column (`'2010 [YR2010]'` format)  
**Country code:** ISO3 (e.g. `AFG`) — already in the correct format  

**Key issues:**
- Missing values encoded as `'..'` → replaced with `NaN`
- Year columns extracted via regex, then melted to long format

**Indicators selected:**
| Column Name | Original Series Name |
|---|---|
| `gdp_per_capita` | GDP per capita (current US$) |
| `population` | Population, total |
| `population_growth_pct` | Population growth (annual %) |
| `precipitation_mm` | Average precipitation in depth (mm per year) |
| `poverty_rate` | Poverty headcount ratio at $3.00 a day (2021 PPP) (%) |

**Excluded:**
- `Droughts, floods, extreme temperatures` — fixed 1990–2009 average, not time-series compatible

In [6]:
WDI_SERIES = [
    "GDP per capita (current US$)",
    "Population, total",
    "Population growth (annual %)",
    "Average precipitation in depth (mm per year)",
    "Poverty headcount ratio at $3.00 a day (2021 PPP) (% of population)",
]

wdi_work = wdi[wdi["Series Name"].isin(WDI_SERIES)].copy()

# Identify year columns using regex ('2010 [YR2010]' format)
year_cols = [c for c in wdi_work.columns if re.match(r"^\d{4} \[YR\d{4}\]$", c)]
year_cols_2010_2022 = [c for c in year_cols if 2010 <= int(c[:4]) <= 2022]
print(f"Year columns: {len(year_cols_2010_2022)} ({year_cols_2010_2022[0]} ... {year_cols_2010_2022[-1]})")

# Keep only necessary columns
wdi_work = wdi_work[["Country Name", "Country Code", "Series Name"] + year_cols_2010_2022].copy()

# Wide -> Long (melt year columns)
wdi_long = wdi_work.melt(
    id_vars=["Country Name", "Country Code", "Series Name"],
    value_vars=year_cols_2010_2022,
    var_name="Year_str",
    value_name="Value"
)

# Extract integer year from '2010 [YR2010]'
wdi_long["Year"] = wdi_long["Year_str"].str[:4].astype(int)
wdi_long = wdi_long.drop(columns="Year_str")

# Replace '..' with NaN, then coerce to numeric
wdi_long["Value"] = wdi_long["Value"].replace("..", np.nan)
wdi_long["Value"] = pd.to_numeric(wdi_long["Value"], errors="coerce")

# Long -> Wide (Series Name becomes columns)
wdi_clean = wdi_long.pivot_table(
    index=["Country Name", "Country Code", "Year"],
    columns="Series Name",
    values="Value"
).reset_index()
wdi_clean.columns.name = None

# Rename columns
wdi_clean = wdi_clean.rename(columns={
    "Country Name":                                                         "Area",
    "Country Code":                                                         "iso3",
    "GDP per capita (current US$)":                                         "gdp_per_capita",
    "Population, total":                                                    "population",
    "Population growth (annual %)":                                         "population_growth_pct",
    "Average precipitation in depth (mm per year)":                        "precipitation_mm",
    "Poverty headcount ratio at $3.00 a day (2021 PPP) (% of population)": "poverty_rate",
})

print(f"WDI cleaned: {wdi_clean.shape}")
print(f"  Countries : {wdi_clean['Area'].nunique()}")
print(f"  Years     : {sorted(wdi_clean['Year'].unique())}")
print(f"  Columns   : {wdi_clean.columns.tolist()}")
print(wdi_clean.head(3).to_string())

Year columns: 13 (2010 [YR2010] ... 2022 [YR2022])
WDI cleaned: (3445, 8)
  Countries : 265
  Years     : [2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022]
  Columns   : ['Area', 'iso3', 'Year', 'precipitation_mm', 'gdp_per_capita', 'population_growth_pct', 'population', 'poverty_rate']
          Area iso3  Year  precipitation_mm  gdp_per_capita  population_growth_pct  population  poverty_rate
0  Afghanistan  AFG  2010             327.0      560.621505               2.934687  28284089.0           NaN
1  Afghanistan  AFG  2011             327.0      606.694676               3.691503  29347708.0           NaN
2  Afghanistan  AFG  2012             327.0      651.417134               4.047863  30560034.0           NaN


---
## Cell 7 — Standardize Country Codes (M49 -> ISO3)

**Problem:** FAOSTAT uses M49 numeric codes; World Bank uses ISO3 alpha codes.  
They must be unified before merging.

**Approach:** Use `pycountry` library to map M49 → ISO3 automatically,  
then handle failures manually.

**Known issue:** M49 code `159` ("China" aggregate) and M49 `156` ("China, mainland")  
both map to ISO3 `CHN` → causes duplicate rows after merge.  
**Solution:** Exclude M49=159 from all FAOSTAT datasets (156 is the primary record).

In [7]:
def m49_to_iso3(m49_code):
    """Convert M49 numeric country code to ISO3 alpha code using pycountry."""
    try:
        country = pycountry.countries.get(numeric=str(m49_code).zfill(3))
        return country.alpha_3 if country else np.nan
    except:
        return np.nan

# Build mapping table from FSI country list
m49_map = fsi_clean[["Area Code (M49)", "Area"]].drop_duplicates()
m49_map["iso3"] = m49_map["Area Code (M49)"].apply(m49_to_iso3)

# Check conversion failures
failed = m49_map[m49_map["iso3"].isna()]
print(f"Automatic conversion failures: {len(failed)}")
print(failed[["Area Code (M49)", "Area"]].to_string())

# Manual fix for M49=159 (will be excluded from datasets later)
m49_map.loc[m49_map["Area Code (M49)"] == 159, "iso3"] = "CHN"
print(f"\nConversion failures after fix: {m49_map['iso3'].isna().sum()}")

Automatic conversion failures: 1
     Area Code (M49)   Area
520              159  China

Conversion failures after fix: 0


In [8]:
# Attach iso3 to all three FAOSTAT datasets
fsi_clean = fsi_clean.merge(m49_map[["Area Code (M49)", "iso3"]], on="Area Code (M49)", how="left")
fb_clean  = fb_clean.merge(m49_map[["Area Code (M49)", "iso3"]],  on="Area Code (M49)", how="left")
pi_clean  = pi_clean.merge(m49_map[["Area Code (M49)", "iso3"]],  on="Area Code (M49)", how="left")

print("After initial iso3 merge:")
print(f"  FSI iso3 missing: {fsi_clean['iso3'].isna().sum()}")
print(f"  FB  iso3 missing: {fb_clean['iso3'].isna().sum()}")
print(f"  PI  iso3 missing: {pi_clean['iso3'].isna().sum()}")

# Manual fixes for small island territories absent from pycountry
manual_map = {
    184: "COK",  # Cook Islands
    234: "FRO",  # Faroe Islands
    570: "NIU",  # Niue
    772: "TKL",  # Tokelau
}
for m49, iso3 in manual_map.items():
    pi_clean.loc[pi_clean["Area Code (M49)"] == m49, "iso3"] = iso3

print(f"\nAfter manual fixes:")
print(f"  PI  iso3 missing: {pi_clean['iso3'].isna().sum()}")
print("All country codes standardized to ISO3")

After initial iso3 merge:
  FSI iso3 missing: 0
  FB  iso3 missing: 0
  PI  iso3 missing: 44

After manual fixes:
  PI  iso3 missing: 0
All country codes standardized to ISO3


---
## Cell 8 — Merge All Datasets

**Strategy:**
1. Exclude M49=159 from all FAOSTAT datasets to eliminate CHN duplicates
2. Merge FSI + FB + PI on `[iso3, Year]` (left join, FSI as base)
3. Merge result with WDI on `[iso3, Year]` (left join)

Left join preserves all FSI country-year combinations,  
accepting NaN where other datasets have no coverage.

In [9]:
# Exclude M49=159 (China aggregate) from all FAOSTAT datasets
# M49=156 (China, mainland) is retained as the primary record
fsi_clean2 = fsi_clean[fsi_clean["Area Code (M49)"] != 159].copy()
fb_clean2  = fb_clean[fb_clean["Area Code (M49)"]   != 159].copy()
pi_clean2  = pi_clean[pi_clean["Area Code (M49)"]   != 159].copy()

# Step 1: Merge three FAOSTAT datasets
faostat = fsi_clean2.merge(
    fb_clean2[["iso3", "Year", "fat_supply_g", "food_supply_kcal", "protein_supply_g"]],
    on=["iso3", "Year"],
    how="left"
).merge(
    pi_clean2[["iso3", "Year", "agri_production_index", "food_production_index"]],
    on=["iso3", "Year"],
    how="left"
)

# Step 2: Merge with WDI
df = faostat.merge(
    wdi_clean[["iso3", "Year", "gdp_per_capita", "population",
               "population_growth_pct", "precipitation_mm", "poverty_rate"]],
    on=["iso3", "Year"],
    how="left"
)

# Verify no duplicate (iso3, Year) pairs
dupes = df[df.duplicated(subset=["iso3", "Year"], keep=False)]
print(f"Duplicate rows: {len(dupes)}")

print(f"\nMerge complete: {df.shape}")
print(f"  Countries : {df['iso3'].nunique()}")
print(f"  Years     : {sorted(df['Year'].unique())}")

Duplicate rows: 0

Merge complete: (2599, 19)
  Countries : 200
  Years     : [2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022]


---
## Cell 9 — Missing Value Summary

In [10]:
print("=== Missing Value Summary ===")
missing = df.isnull().mean().mul(100).round(1)
missing = missing[missing > 0].sort_values(ascending=False)
print(missing.to_string())
print()
print("Decision: All columns retained as-is.")
print("Missing values will be addressed in Phase 4 (Modeling) as needed.")
print()
print("Notes:")
print("  poverty_rate (60.1%)         : High missingness is a known limitation; retained for analysis")
print("  undernourishment_pct (22.6%) : Target variable — NOT dropped")
print("  agri/food_production (22.0%) : Source-level gaps in PI (years 2015, 2018 absent)")

=== Missing Value Summary ===
poverty_rate                    60.1
undernourishment_pct            22.6
cereal_import_dependency_pct    22.4
agri_production_index           22.0
food_production_index           22.0
food_supply_variability_kcal    16.4
dietary_energy_supply_kcal      16.2
fat_supply_g                    14.0
food_supply_kcal                14.0
protein_supply_g                14.0
precipitation_mm                 9.2
political_stability_index        2.5
gdp_per_capita                   1.9
population                       0.5
population_growth_pct            0.5

Decision: All columns retained as-is.
Missing values will be addressed in Phase 4 (Modeling) as needed.

Notes:
  poverty_rate (60.1%)         : High missingness is a known limitation; retained for analysis
  undernourishment_pct (22.6%) : Target variable — NOT dropped
  agri/food_production (22.0%) : Source-level gaps in PI (years 2015, 2018 absent)


---
## Cell 10 — Reorder Columns & Save to Processed

In [11]:
# Reorder columns by category
col_order = [
    # Keys
    "iso3", "Area", "Year",
    # Target variable
    "undernourishment_pct",
    # Food supply
    "dietary_energy_supply_kcal", "food_supply_kcal",
    "protein_supply_g", "fat_supply_g",
    "food_supply_variability_kcal", "cereal_import_dependency_pct",
    # Agricultural production
    "agri_production_index", "food_production_index",
    # Economic
    "gdp_per_capita", "poverty_rate",
    # Demographic
    "population", "population_growth_pct",
    # Climate
    "precipitation_mm",
    # Political
    "political_stability_index",
]

df = df[col_order]

# Save all files to data/processed/
os.makedirs("../data/processed", exist_ok=True)

df.to_csv("../data/processed/merged_final.csv", index=False)
fsi_clean2.to_csv("../data/processed/fsi_clean.csv", index=False)
fb_clean2.to_csv("../data/processed/fb_clean.csv",   index=False)
pi_clean2.to_csv("../data/processed/pi_clean.csv",   index=False)
wdi_clean.to_csv("../data/processed/wdi_clean.csv",  index=False)

print("All files saved to data/processed/")
print(f"  merged_final.csv : {df.shape}")
print(f"  fsi_clean.csv    : {fsi_clean2.shape}")
print(f"  fb_clean.csv     : {fb_clean2.shape}")
print(f"  pi_clean.csv     : {pi_clean2.shape}")
print(f"  wdi_clean.csv    : {wdi_clean.shape}")
print()
print("=== Final Dataset Preview ===")
print(df.head(5).to_string())

All files saved to data/processed/
  merged_final.csv : (2599, 18)
  fsi_clean.csv    : (2599, 9)
  fb_clean.csv     : (2235, 7)
  pi_clean.csv     : (2071, 6)
  wdi_clean.csv    : (3445, 8)

=== Final Dataset Preview ===
  iso3         Area  Year  undernourishment_pct  dietary_energy_supply_kcal  food_supply_kcal  protein_supply_g  fat_supply_g  food_supply_variability_kcal  cereal_import_dependency_pct  agri_production_index  food_production_index  gdp_per_capita  poverty_rate  population  population_growth_pct  precipitation_mm  political_stability_index
0  AFG  Afghanistan  2010                   NaN                      2200.0           2200.21             65.54         37.93                          28.0                           NaN                 109.33                 109.42      560.621505           NaN  28284089.0               2.934687             327.0                      -2.58
1  AFG  Afghanistan  2011                  17.7                      2172.0           2171.86 

---
## Phase 2 Summary

| Item | Result |
|---|---|
| Final shape | (2599, 18) |
| Countries | 200 |
| Years | 2010–2022 |
| Duplicate rows | 0 |
| Output file | `data/processed/merged_final.csv` |

### Issues Encountered & Resolved

| Issue | Root Cause | Resolution |
|---|---|---|
| BOM character in column names | FAOSTAT UTF-8-BOM encoding | `encoding='utf-8-sig'` |
| `Year` as string object | 3-year average indicators use range format `'2009-2011'` | `parse_year()` extracts end year |
| `Value` column non-numeric | `'<2.5'` detection limit string | Replaced with `2.5` (upper bound) |
| Duplicate rows for China | M49=159 (aggregate) and M49=156 (mainland) both map to ISO3=CHN | Excluded M49=159 from all FAOSTAT datasets |
| 4 countries missing ISO3 | Small territories absent from pycountry | Manually mapped COK, FRO, NIU, TKL |
| PI missing years 2015, 2018 | Absent from source data | Confirmed as source-level gap, not a bug |

### Next Step: Phase 3 — Exploratory Data Analysis (`03_eda.ipynb`)
- Distribution of `undernourishment_pct` (target variable)
- Correlation heatmap across all features
- Country and regional comparisons
- Time-series trends (2010–2022)